# Submission 2 \\ Multi-Head Attention
This notebook loads a pre-trained NumPy model with a multi-head attention block,
runs it on the evaluation set (datasets/evaluation/subm2.csv), and writes a
submission file subm2-g5-MEI-A.csv with vendor labels (Human, OpenAI, Google, Anthropic, Meta).

In [ ]:
import os
import sys
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path("/home/local/private/repos/DL-AITextClassifier").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.dataloader.dataloader import SentenceDataModule, MODEL_MAP
from src.models.gru_numpy import (
    DenseLayer,
    MultiHeadAttentionLayer,
    EmbeddingLayer,
    SequenceMeanMaxPool,
    DropoutLayer,
    NeuralNetwork,
    CrossEntropyLoss,
    SoftmaxActivation,
    accuracy,
)
from src.text_embedding import NgramEmbedding

np.random.seed(42)

import pandas as pd

ModuleNotFoundError: No module named 'src'

In [ ]:
eval_path = PROJECT_ROOT / "datasets" / "evaluation" / "subm2.csv"
subm2_df = pd.read_csv(eval_path, sep=";")
subm2_df.head()

In [ ]:
id_to_model = {v: k for k, v in MODEL_MAP.items()}
MODEL_MAP_VENDOR = {
    0: "Human",
    1: "OpenAI",
    2: "Google",
    3: "Anthropic",
    4: "Meta",
}

texts_eval = subm2_df["Text"].tolist()
ids_eval = subm2_df["ID"].tolist()
len(texts_eval), texts_eval[0][:200]

In [ ]:
from typing import Iterator, Tuple, List

def batch_iterator(X: List[str], y: np.ndarray | None = None, batch_size: int = 128,
                   shuffle: bool = False) -> Iterator[Tuple[List[str], np.ndarray | None]]:
    indices = np.arange(len(X))
    if shuffle:
        np.random.shuffle(indices)
    for start in range(0, len(indices), batch_size):
        batch_idx = indices[start:start + batch_size]
        batch_X = [X[i] for i in batch_idx]
        if y is not None:
            batch_y = y[batch_idx]
        else:
            batch_y = None
        yield batch_X, batch_y

for bx, _ in batch_iterator(texts_eval, batch_size=4):
    print(len(bx), bx[0][:80])
    break

In [ ]:
seed = 42
record_path = PROJECT_ROOT / "datasets" / "records_long.json"
weights_path = PROJECT_ROOT / "cache" / "numpy_mha_trained.pkl"
current_epoch_weights_path = PROJECT_ROOT / "cache" / "numpy_mha_current_epoch.pkl"
best_epoch_weights_path = PROJECT_ROOT / "cache" / "numpy_mha_best_epoch.pkl"

dm = SentenceDataModule(record_path=str(record_path), size=100000, split=(70, 20, 10), seed=seed)
embedding_size = 128
embedding = NgramEmbedding(dm.get_most_common_words(top_n=30000), embedding_size, seed=seed)

In [ ]:
loaders = {
    "train": dm.get_train_loader(),
    "val": dm.get_val_loader(),
    "test": dm.get_test_loader(),
}

for name, loader in loaders.items():
    print(f"=== {name.upper()} SAMPLES ===")
    for i in range(min(3, len(loader))):
        print(loader[i])
    print()

In [ ]:
net = NeuralNetwork(
    dm,
    epochs=100,
    batch_size=32,
    learning_rate=0.01,
    verbose=True,
    loss=CrossEntropyLoss,
    metric=accuracy,
    early_stopping_patience=10,
    lr_patience=4,
    lr_decay_factor=0.5,
    min_learning_rate=1e-5,
    num_train_epoch_batches=100,
    num_val_epoch_batches=10,
    save_current_epoch_weights_path=str(current_epoch_weights_path),
    save_best_epoch_weights_path=str(best_epoch_weights_path),
)

net.add(EmbeddingLayer(embedding=embedding))
net.add(MultiHeadAttentionLayer(num_heads=4, model_dim=embedding_size, seed=seed))
net.add(SequenceMeanMaxPool())
net.add(DropoutLayer(rate=0.3, seed=seed))
net.add(DenseLayer(5))
net.add(SoftmaxActivation())

In [ ]:
net.load_weights(str(weights_path))

In [ ]:
out = net.predict()
print(net.score(out))

In [ ]:
submission_path = PROJECT_ROOT / "subm2" / "subm2-g5-MEI-A.csv"

eval_texts = subm2_df["Text"].tolist()

from src.models.gru_numpy.neuralnet import ID_TO_VENDOR

def predict_texts(model: NeuralNetwork, texts, batch_size: int = 32):
    preds = []
    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]
        preds.append(model.forward_propagation(batch, training=False))
    return np.concatenate(preds)

outputs = predict_texts(net, eval_texts, batch_size=32)

if outputs.ndim == 1:
    pred_ids = (outputs > 0.5).astype(int)
else:
    pred_ids = np.argmax(outputs, axis=1)

labels = [ID_TO_VENDOR.get(int(idx), "Unknown") for idx in pred_ids]

submission_path.parent.mkdir(parents=True, exist_ok=True)
subm2_df_out = pd.DataFrame({"ID": subm2_df["ID"], "Label": labels})
subm2_df_out.to_csv(submission_path, index=False)
submission_path, subm2_df_out.head()